**Anggota Kelompok:**

1.   Isrina Luthfiah (2409116003)
2.   Nayla Camelia Indraswari (2409116009)
3.   Muhammad Sadikin Samir (2409116031)


**Sumber Data**: Retail Insights Sales, Inventory & Customer etc from **Kaggle**

 https://www.kaggle.com/datasets/carebymanu/retail-insights-sales-inventory-and-customer-etc

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import re
import zipfile
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

RUN_NAME = f"pantau_etl_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# ETL RUNTIME METADATA
ETL_START_TIME = datetime.now()
ETL_MODE = "full_refresh"
SOURCE_SYSTEM = "Kaggle Retail Insights Dataset"
TARGET_SYSTEM = "Pantau BI Data Warehouse Prototype"


Langkah pertama yang dilakukan adalah menyiapkan library yang dibutuhkan untuk proses data warehouse.

### **helper function**

In [ ]:
def snake_case(name: str) -> str:
    """Convert column names to snake_case."""
    name = str(name).strip()
    name = re.sub(r"[^0-9a-zA-Z]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_").lower()


def read_csv_clean(path: Path) -> pd.DataFrame:
    """Read CSV and standardize column names."""
    df = pd.read_csv(path)
    df.columns = [snake_case(c) for c in df.columns]
    return df


def parse_date_series(series: pd.Series, dayfirst: bool = True) -> pd.Series:
    """Robust date parser."""
    return pd.to_datetime(series, errors="coerce", dayfirst=dayfirst).dt.date


def month_name_from_num(month_num):
    if pd.isna(month_num):
        return None
    return pd.Timestamp(year=2000, month=int(month_num), day=1).strftime("%B")


def safe_divide(numerator, denominator, default=0):
    denominator = denominator.replace(0, np.nan) if isinstance(denominator, pd.Series) else denominator
    result = numerator / denominator
    if isinstance(result, pd.Series):
        return result.replace([np.inf, -np.inf], np.nan).fillna(default)
    return default if denominator in [0, None] or pd.isna(denominator) else result


def value_segment(series: pd.Series) -> pd.Series:
    """Segment customer/product monetary value using quantiles."""
    if series.nunique() < 3:
        return pd.Series(["medium_value"] * len(series), index=series.index)
    q1 = series.quantile(0.33)
    q2 = series.quantile(0.66)
    return np.select(
        [series <= q1, (series > q1) & (series <= q2), series > q2],
        ["low_value", "medium_value", "high_value"],
        default="medium_value"
    )


def stock_status_rule(row):
    """Classify stock condition using ending inventory, sales, and stockout flag."""
    ending = row.get("total_ending_inventory", row.get("ending_inventory", 0))
    sold = row.get("total_units_sold", row.get("units_sold", 0))
    stockout_count = row.get("stockout_count", 0)
    ratio = row.get("sales_to_stock_ratio", 0)

    if str(row.get("stockout_flag", "")).lower() == "yes" or ending <= 0 or stockout_count > 0:
        return "out_of_stock"
    if ending <= 20 or ratio >= 1.0 or ending <= sold * 0.25:
        return "low_stock"
    if ending >= sold * 3 and ending > 100:
        return "overstock"
    return "safe"


def restock_priority_rule(row):
    status = row.get("stock_status", "safe")
    ratio = row.get("sales_to_stock_ratio", 0)
    if status == "out_of_stock":
        return "urgent"
    if status == "low_stock" and ratio >= 1:
        return "high"
    if status == "low_stock":
        return "medium"
    if status == "overstock":
        return "no_restock"
    return "normal"

def create_column_profile(table_name: str, df: pd.DataFrame) -> pd.DataFrame:
    """Create column-level profiling for data understanding documentation."""
    return pd.DataFrame({
        "table_name": table_name,
        "column_name": df.columns,
        "data_type": [str(dtype) for dtype in df.dtypes],
        "total_rows": len(df),
        "missing_values": df.isna().sum().values,
        "missing_percentage": (df.isna().sum().values / max(len(df), 1) * 100).round(2),
        "unique_values": df.nunique(dropna=True).values,
    })


def check_required_columns(table_name: str, df: pd.DataFrame, required_columns: list) -> pd.DataFrame:
    """Check whether required columns exist in a table."""
    rows = []
    for col in required_columns:
        rows.append({
            "table_name": table_name,
            "check_type": "required_column",
            "column_name": col,
            "value": "exists" if col in df.columns else "missing",
            "status": "ok" if col in df.columns else "failed"
        })
    return pd.DataFrame(rows)


def check_null_keys(table_name: str, df: pd.DataFrame, key_columns: list) -> pd.DataFrame:
    """Check null values in key columns."""
    rows = []
    for col in key_columns:
        if col in df.columns:
            null_count = int(df[col].isna().sum())
            rows.append({
                "table_name": table_name,
                "check_type": "null_key",
                "column_name": col,
                "value": null_count,
                "status": "ok" if null_count == 0 else "failed"
            })
    return pd.DataFrame(rows)


def check_numeric_quality(table_name: str, df: pd.DataFrame, numeric_columns: list, allow_negative: bool = False) -> pd.DataFrame:
    """Check missing and negative values in numeric columns."""
    rows = []
    for col in numeric_columns:
        if col in df.columns:
            missing_count = int(df[col].isna().sum())
            if pd.api.types.is_numeric_dtype(df[col]):
                negative_count = int((df[col] < 0).sum())
            else:
                negative_count = 0

            rows.append({
                "table_name": table_name,
                "check_type": "numeric_missing",
                "column_name": col,
                "value": missing_count,
                "status": "ok" if missing_count == 0 else "warning"
            })
            rows.append({
                "table_name": table_name,
                "check_type": "numeric_negative",
                "column_name": col,
                "value": negative_count,
                "status": "ok" if allow_negative or negative_count == 0 else "warning"
            })
    return pd.DataFrame(rows)


def make_source_to_target_mapping() -> pd.DataFrame:
    """Create source-to-target mapping documentation for ETL reporting."""
    rows = [
        {
            "source_table": "Sales_Data",
            "source_column": "date",
            "target_table": "dw_dim_date / dw_fact_sales",
            "target_column": "date_key",
            "transformation": "parse date, convert to YYYYMMDD integer"
        },
        {
            "source_table": "Sales_Data",
            "source_column": "product_id, customer_id, site_id",
            "target_table": "dw_fact_sales",
            "target_column": "product_key, customer_key, site_key",
            "transformation": "map natural keys to surrogate keys"
        },
        {
            "source_table": "Sales_Data",
            "source_column": "units_sold, revenue, discounts, returns",
            "target_table": "dw_fact_sales / final_sales_dataset",
            "target_column": "units_sold, revenue, discounts, returns, net_revenue, estimated_profit",
            "transformation": "numeric conversion and derived metrics based on the original ETL logic"
        },
        {
            "source_table": "Product_Information",
            "source_column": "product_id, product_name, category, subcategory, supplier, unit_cost, unit_price",
            "target_table": "dw_dim_product",
            "target_column": "product attributes",
            "transformation": "standardize column names, clean text, numeric conversion"
        },
        {
            "source_table": "Customer_Demographics",
            "source_column": "customer_id, age, gender, income_bracket, purchase_frequency, average_spend",
            "target_table": "dw_dim_customer / customer_analysis",
            "target_column": "customer attributes and customer segments",
            "transformation": "numeric conversion, segmentation by value and frequency"
        },
        {
            "source_table": "Site_Details",
            "source_column": "site_id, site_name, region, city, state, store_size, open_date",
            "target_table": "dw_dim_site",
            "target_column": "site attributes",
            "transformation": "parse open_date, standardize text/numeric columns"
        },
        {
            "source_table": "Promotions_and_Discounts",
            "source_column": "promotion_id, product_id, site_id, start_date, end_date, discount_type, discount_amount",
            "target_table": "dw_dim_promotion / dw_fact_promotion / promotion_performance",
            "target_column": "promotion attributes and promotion measures",
            "transformation": "parse campaign period, classify status, match active promo to sales"
        },
        {
            "source_table": "Inventory_Data",
            "source_column": "product_id, site_id, beginning_inventory, ending_inventory, replenishment, stockout_flag",
            "target_table": "dw_fact_inventory / inventory_monitoring",
            "target_column": "inventory measures and stock status",
            "transformation": "calculate inventory_change, stock_status, and restock_priority"
        },
        {
            "source_table": "Logistics_Data",
            "source_column": "shipment_id, product_id, site_id, shipment_date, quantity, delivery_status, transportation_type",
            "target_table": "dw_fact_logistics / logistics_performance",
            "target_column": "logistics measures and delivery performance",
            "transformation": "parse shipment_date and aggregate delivery status"
        },
        {
            "source_table": "Monthly_Seasonal_Planning",
            "source_column": "month, product_category, forecasted_sales, actual_sales, seasonal_adjustments",
            "target_table": "dw_fact_seasonal_planning / seasonal_planning_performance",
            "target_column": "forecast, actual, error, accuracy",
            "transformation": "parse month and calculate forecast error/accuracy"
        },
    ]
    return pd.DataFrame(rows)


def make_fact_grain_documentation() -> pd.DataFrame:
    """Create fact table grain documentation for academic reporting."""
    rows = [
        {
            "fact_table": "dw_fact_sales",
            "grain": "One row represents one sales record for a product, customer, site, date, and active promotion if available.",
            "main_measures": "units_sold, revenue, discounts, returns, net_revenue, estimated_profit"
        },
        {
            "fact_table": "dw_fact_inventory",
            "grain": "One row represents one inventory snapshot for a product at a site.",
            "main_measures": "beginning_inventory, ending_inventory, replenishment, inventory_change"
        },
        {
            "fact_table": "dw_fact_logistics",
            "grain": "One row represents one shipment activity for a product and site.",
            "main_measures": "quantity"
        },
        {
            "fact_table": "dw_fact_promotion",
            "grain": "One row represents one promotion campaign for a product and site.",
            "main_measures": "discount_amount"
        },
        {
            "fact_table": "dw_fact_seasonal_planning",
            "grain": "One row represents one monthly seasonal planning record for a product category and site if available.",
            "main_measures": "forecasted_sales, actual_sales, seasonal_adjustments"
        },
    ]
    return pd.DataFrame(rows)


Langkah selanjutnya ialah membuat *helper function* yang bisa dipakai berulang selama proses ETL. Fungsinya antara lain menstandarkan nama kolom, membaca file CSV, mengubah format tanggal, melakukan pembagian aman, membuat segmentasi customer, menentukan status stok, menentukan prioritas restock, serta menyiapkan pengecekan kualitas data. Tujuannya agar proses transformasi lebih konsisten dan tidak perlu menulis kode secara berulang.

## **EXTRACT**

### **Extract Data**

In [ ]:
#read dataset

DATA_PATH = "/content/drive/MyDrive/TUGAS SMESTER 4/BI/dataset"

sales_raw = read_csv_clean(f"{DATA_PATH}/Sales_Data.csv")
products_raw = read_csv_clean(f"{DATA_PATH}/Product_Information.csv")
inventory_raw = read_csv_clean(f"{DATA_PATH}/Inventory_Data.csv")
customers_raw = read_csv_clean(f"{DATA_PATH}/Customer_Demographics.csv")
sites_raw = read_csv_clean(f"{DATA_PATH}/Site_Details.csv")
logistics_raw = read_csv_clean(f"{DATA_PATH}/Logistics_Data.csv")
promos_raw = read_csv_clean(f"{DATA_PATH}/Promotions_and_Discounts.csv")
planning_raw = read_csv_clean(f"{DATA_PATH}/Monthly_Seasonal_Planning.csv")

raw_tables = {
    "sales_raw": sales_raw,
    "products_raw": products_raw,
    "inventory_raw": inventory_raw,
    "customers_raw": customers_raw,
    "sites_raw": sites_raw,
    "logistics_raw": logistics_raw,
    "promos_raw": promos_raw,
    "planning_raw": planning_raw,
}

print("\nRaw table shapes:")
for name, df in raw_tables.items():
    print(f"{name}: {df.shape}")


Raw table shapes:
sales_raw: (219, 8)
products_raw: (100, 8)
inventory_raw: (100, 6)
customers_raw: (50, 6)
sites_raw: (50, 9)
logistics_raw: (150, 7)
promos_raw: (50, 7)
planning_raw: (120, 6)


Proses membaca seluruh dataset sumber dari folder Google Drive. Dataset yang dibaca meliputi sales, product, inventory, customer, site, logistics, promotions, dan planning. Semua data dikumpulkan dalam `raw_tables`.

## **TRANSFORM**

### **Data understanding**

In [ ]:
# Data understanding
profile_rows = []
for name, df in raw_tables.items():
    profile_rows.append({
        "table_name": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_values": int(df.isna().sum().sum()),
    })
profile_df = pd.DataFrame(profile_rows)
print("\nData profiling summary:")
display(profile_df) if "display" in globals() else print(profile_df)


Data profiling summary:
      table_name  rows  columns  duplicate_rows  missing_values
0      sales_raw   219        8               0               0
1   products_raw   100        8               0               0
2  inventory_raw   100        6               0               0
3  customers_raw    50        6               0               0
4      sites_raw    50        9               0               0
5  logistics_raw   150        7               0               0
6     promos_raw    50        7               0               0
7   planning_raw   120        6               0               0


Data understanding dilakukan dengan membuat ringkasan awal untuk setiap tabel. Kode menghitung jumlah baris, jumlah kolom, jumlah baris duplikat, dan total missing values. Tujuannya untuk mengetahui kondisi data sebelum masuk ke tahap *cleaning* dan transformasi.

### **Data cleaning**

In [ ]:
# Data Cleaning

sales = sales_raw.drop_duplicates().copy()
products = products_raw.drop_duplicates().copy()
inventory = inventory_raw.drop_duplicates().copy()
customers = customers_raw.drop_duplicates().copy()
sites = sites_raw.drop_duplicates().copy()
logistics = logistics_raw.drop_duplicates().copy()
promos = promos_raw.drop_duplicates().copy()
planning = planning_raw.drop_duplicates().copy()

# Date parsing
sales["sales_date"] = parse_date_series(sales["date"], dayfirst=True)
logistics["shipment_date"] = parse_date_series(logistics["shipment_date"], dayfirst=True)
promos["start_date"] = parse_date_series(promos["start_date"], dayfirst=True)
promos["end_date"] = parse_date_series(promos["end_date"], dayfirst=True)
sites["open_date"] = parse_date_series(sites["open_date"], dayfirst=True)
planning["month_date"] = pd.to_datetime(planning["month"], format="%b-%Y", errors="coerce").dt.date

# Numeric cleaning
numeric_specs = {
    "sales": ["units_sold", "revenue", "discounts", "returns"],
    "products": ["unit_cost", "unit_price", "shelf_life"],
    "inventory": ["beginning_inventory", "ending_inventory", "replenishment"],
    "customers": ["age", "purchase_frequency", "average_spend"],
    "sites": ["store_size"],
    "logistics": ["quantity"],
    "promos": ["discount_amount"],
    "planning": ["forecasted_sales", "actual_sales", "seasonal_adjustments"],
}
for df_name, cols in numeric_specs.items():
    df = locals()[df_name]
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Fill text nulls
for df in [sales, products, inventory, customers, sites, logistics, promos, planning]:
    text_cols = df.select_dtypes(include="object").columns
    for col in text_cols:
        df[col] = df[col].fillna("Unknown").astype(str).str.strip()

# Derived date fields in sales
sales["sales_date_dt"] = pd.to_datetime(sales["sales_date"])
sales["year"] = sales["sales_date_dt"].dt.year
sales["quarter"] = sales["sales_date_dt"].dt.quarter
sales["month"] = sales["sales_date_dt"].dt.month
sales["month_name"] = sales["sales_date_dt"].dt.month_name()
sales["day"] = sales["sales_date_dt"].dt.day
sales["day_name"] = sales["sales_date_dt"].dt.day_name()
sales.drop(columns=["sales_date_dt"], inplace=True)

Data cleaning merupakan proses membersihkan dan menstandarkan data. Bagian ini mencakup proses menghapus duplikasi, mengubah kolom tanggal ke format date, mengonversi kolom numerik agar aman untuk perhitungan, membersihkan kolom teks, dan membuat atribut waktu dari data sales. Tujuannya agar seluruh dataset punya format yang konsisten sebelum dibentuk menjadi tabel warehouse dan data mart.

### **Build dimension tables**

In [ ]:

# dim_date
date_values = []
for s in [sales["sales_date"], logistics["shipment_date"], promos["start_date"], promos["end_date"], planning["month_date"]]:
    date_values.extend(list(pd.Series(s).dropna().unique()))

dim_date = pd.DataFrame({"full_date": sorted(pd.to_datetime(pd.Series(date_values)).dropna().dt.date.unique())})
dim_date["date_key"] = pd.to_datetime(dim_date["full_date"]).dt.strftime("%Y%m%d").astype(int)
dim_date["day"] = pd.to_datetime(dim_date["full_date"]).dt.day
dim_date["month"] = pd.to_datetime(dim_date["full_date"]).dt.month
dim_date["month_name"] = pd.to_datetime(dim_date["full_date"]).dt.month_name()
dim_date["quarter"] = pd.to_datetime(dim_date["full_date"]).dt.quarter
dim_date["year"] = pd.to_datetime(dim_date["full_date"]).dt.year
dim_date["day_name"] = pd.to_datetime(dim_date["full_date"]).dt.day_name()

# dim_product
dim_product = products.copy()
dim_product.insert(0, "product_key", range(1, len(dim_product) + 1))

# dim_customer
dim_customer = customers.copy()
dim_customer.insert(0, "customer_key", range(1, len(dim_customer) + 1))
dim_customer["customer_value_segment"] = value_segment(dim_customer["average_spend"])
dim_customer["buying_frequency_segment"] = np.select(
    [dim_customer["purchase_frequency"] <= 3,
     (dim_customer["purchase_frequency"] > 3) & (dim_customer["purchase_frequency"] <= 7),
     dim_customer["purchase_frequency"] > 7],
    ["low_frequency", "medium_frequency", "high_frequency"],
    default="medium_frequency"
)

# dim_site
dim_site = sites.copy()
dim_site.insert(0, "site_key", range(1, len(dim_site) + 1))

# dim_promotion
dim_promotion = promos.copy()
dim_promotion.insert(0, "promotion_key", range(1, len(dim_promotion) + 1))
dim_promotion["promotion_status"] = np.where(
    pd.to_datetime(dim_promotion["end_date"]) < pd.Timestamp.today().normalize(),
    "ended",
    "active_or_future"
)

# Keys map
product_key_map = dim_product.set_index("product_id")["product_key"].to_dict()
customer_key_map = dim_customer.set_index("customer_id")["customer_key"].to_dict()
site_key_map = dim_site.set_index("site_id")["site_key"].to_dict()
promo_key_map = dim_promotion.set_index("promotion_id")["promotion_key"].to_dict()
date_key_map = dim_date.set_index("full_date")["date_key"].to_dict()

Pembentukkan dimension table adalah proses membuat tabel penjelas untuk data warehouse yang akan memberikan konteks pada fact table. `dim_date` dibuat dari seluruh tanggal penting, `dim_product` dari data produk, `dim_customer` dari data customer, `dim_site` dari data cabang, dan `dim_promotion` dari data promosi. Setelah itu dibuat key mapping agar natural key seperti `product_id` dan `site_id` bisa dipetakan ke surrogate key seperti `product_key` dan `site_key`.

### **Build fact tables**

In [ ]:
promos_for_lookup = dim_promotion[[
    "promotion_key", "promotion_id", "product_id", "site_id", "start_date", "end_date", "discount_type", "discount_amount"
]].copy()
promos_for_lookup["start_date_dt"] = pd.to_datetime(promos_for_lookup["start_date"])
promos_for_lookup["end_date_dt"] = pd.to_datetime(promos_for_lookup["end_date"])

def find_active_promo(row):
    sale_date = pd.to_datetime(row["sales_date"])
    subset = promos_for_lookup[
        (promos_for_lookup["product_id"] == row["product_id"]) &
        (promos_for_lookup["site_id"] == row["site_id"]) &
        (promos_for_lookup["start_date_dt"] <= sale_date) &
        (promos_for_lookup["end_date_dt"] >= sale_date)
    ]
    if len(subset) == 0:
        return pd.Series({
            "promotion_key": np.nan,
            "active_promotion_id": None,
            "discount_type": None,
            "promotion_discount_amount": 0.0,
        })
    selected = subset.iloc[0]
    return pd.Series({
        "promotion_key": selected["promotion_key"],
        "active_promotion_id": selected["promotion_id"],
        "discount_type": selected["discount_type"],
        "promotion_discount_amount": selected["discount_amount"],
    })

active_promo_cols = sales.apply(find_active_promo, axis=1)
sales_enriched = pd.concat([sales.reset_index(drop=True), active_promo_cols.reset_index(drop=True)], axis=1)

# fact_sales
fact_sales = sales_enriched.copy()
fact_sales.insert(0, "sales_key", range(1, len(fact_sales) + 1))
fact_sales["date_key"] = fact_sales["sales_date"].map(date_key_map)
fact_sales["product_key"] = fact_sales["product_id"].map(product_key_map)
fact_sales["customer_key"] = fact_sales["customer_id"].map(customer_key_map)
fact_sales["site_key"] = fact_sales["site_id"].map(site_key_map)
fact_sales["net_revenue"] = fact_sales["revenue"] - fact_sales["discounts"]

fact_sales = fact_sales.merge(products[["product_id", "unit_cost", "unit_price"]], on="product_id", how="left")
fact_sales["estimated_profit"] = ((fact_sales["unit_price"] - fact_sales["unit_cost"]) * fact_sales["units_sold"]) - fact_sales["discounts"]

fact_sales = fact_sales[[
    "sales_key", "date_key", "sales_date", "product_key", "customer_key", "site_key", "promotion_key",
    "product_id", "customer_id", "site_id", "active_promotion_id",
    "units_sold", "revenue", "discounts", "returns", "net_revenue", "estimated_profit"
]]

# fact_inventory
fact_inventory = inventory.copy()
fact_inventory.insert(0, "inventory_key", range(1, len(fact_inventory) + 1))
fact_inventory["product_key"] = fact_inventory["product_id"].map(product_key_map)
fact_inventory["site_key"] = fact_inventory["site_id"].map(site_key_map)
fact_inventory["inventory_change"] = fact_inventory["ending_inventory"] - fact_inventory["beginning_inventory"]
fact_inventory["stockout_flag_bool"] = fact_inventory["stockout_flag"].str.lower().eq("yes")
fact_inventory["stock_status"] = fact_inventory.apply(stock_status_rule, axis=1)

# fact_logistics
fact_logistics = logistics.copy()
fact_logistics.insert(0, "logistics_key", range(1, len(fact_logistics) + 1))
fact_logistics["shipment_date_key"] = fact_logistics["shipment_date"].map(date_key_map)
fact_logistics["product_key"] = fact_logistics["product_id"].map(product_key_map)
fact_logistics["site_key"] = fact_logistics["site_id"].map(site_key_map)
fact_logistics["delivered_flag"] = fact_logistics["delivery_status"].str.lower().eq("delivered")
fact_logistics["delayed_flag"] = fact_logistics["delivery_status"].str.lower().eq("delayed")
fact_logistics["cancelled_flag"] = fact_logistics["delivery_status"].str.lower().eq("cancelled")

# fact_promotion
fact_promotion = dim_promotion.copy()
fact_promotion.insert(0, "promotion_fact_key", range(1, len(fact_promotion) + 1))
fact_promotion["product_key"] = fact_promotion["product_id"].map(product_key_map)
fact_promotion["site_key"] = fact_promotion["site_id"].map(site_key_map)
fact_promotion["start_date_key"] = fact_promotion["start_date"].map(date_key_map)
fact_promotion["end_date_key"] = fact_promotion["end_date"].map(date_key_map)

# fact_seasonal_planning
fact_seasonal_planning = planning.copy()
fact_seasonal_planning.insert(0, "planning_key", range(1, len(fact_seasonal_planning) + 1))
fact_seasonal_planning["month_key"] = fact_seasonal_planning["month_date"].map(date_key_map)
fact_seasonal_planning["site_key"] = fact_seasonal_planning["site_id"].map(site_key_map)
fact_seasonal_planning["forecast_error"] = fact_seasonal_planning["actual_sales"] - fact_seasonal_planning["forecasted_sales"]
fact_seasonal_planning["forecast_accuracy"] = 100 - (abs(fact_seasonal_planning["forecast_error"]) / fact_seasonal_planning["forecasted_sales"].replace(0, np.nan) * 100)
fact_seasonal_planning["forecast_accuracy"] = fact_seasonal_planning["forecast_accuracy"].replace([np.inf, -np.inf], np.nan).fillna(0)
fact_seasonal_planning["planning_status"] = np.where(
    abs(fact_seasonal_planning["forecast_error"]) <= fact_seasonal_planning["forecasted_sales"] * 0.1,
    "accurate",
    "needs_review"
)

Sebelum membuat `fact_sales`, data sales dicocokkan dulu dengan promosi aktif berdasarkan produk, site, dan periode tanggal. Setelah itu dibentuk `fact_sales`, `fact_inventory`, `fact_logistics`, `fact_promotion`, dan `fact_seasonal_planning`. Tujuannya agar setiap proses bisnis memiliki tabel fakta sesuai grain/konteks data masing-masing.

### **data mart**

In [ ]:
# =========================================================

# 8.1 final_sales_dataset: sales + product + customer + site + active promo + inventory
inventory_for_join = inventory[[
    "site_id", "product_id", "beginning_inventory", "ending_inventory", "replenishment", "stockout_flag"
]].copy()
inventory_for_join["inventory_change"] = inventory_for_join["ending_inventory"] - inventory_for_join["beginning_inventory"]
inventory_for_join["stock_status"] = inventory_for_join.apply(stock_status_rule, axis=1)

final_sales_dataset = sales_enriched.merge(products, on="product_id", how="left")
final_sales_dataset = final_sales_dataset.merge(customers, on="customer_id", how="left")
final_sales_dataset = final_sales_dataset.merge(sites, on="site_id", how="left")
final_sales_dataset = final_sales_dataset.merge(inventory_for_join, on=["site_id", "product_id"], how="left")

final_sales_dataset["net_revenue"] = final_sales_dataset["revenue"] - final_sales_dataset["discounts"]
final_sales_dataset["estimated_profit"] = ((final_sales_dataset["unit_price"] - final_sales_dataset["unit_cost"]) * final_sales_dataset["units_sold"]) - final_sales_dataset["discounts"]

final_sales_dataset = final_sales_dataset[[
    "sales_date", "year", "quarter", "month", "month_name", "day", "day_name",
    "site_id", "site_name", "site_format", "region", "city", "state", "store_size", "status",
    "product_id", "product_name", "category", "subcategory", "unit_cost", "unit_price", "supplier", "shelf_life",
    "customer_id", "age", "gender", "income_bracket", "purchase_frequency", "average_spend",
    "units_sold", "revenue", "discounts", "returns", "net_revenue", "estimated_profit",
    "active_promotion_id", "discount_type", "promotion_discount_amount",
    "beginning_inventory", "ending_inventory", "replenishment", "stockout_flag", "inventory_change", "stock_status"
]].copy()

# 8.2 owner_kpi_summary
owner_kpi_summary = pd.DataFrame([{
    "period_type": "all_time",
    "period_start": final_sales_dataset["sales_date"].min(),
    "period_end": final_sales_dataset["sales_date"].max(),
    "total_sales_records": int(len(final_sales_dataset)),
    "total_units_sold": int(final_sales_dataset["units_sold"].sum()),
    "total_revenue": float(final_sales_dataset["revenue"].sum()),
    "total_discounts": float(final_sales_dataset["discounts"].sum()),
    "total_net_revenue": float(final_sales_dataset["net_revenue"].sum()),
    "estimated_profit": float(final_sales_dataset["estimated_profit"].sum()),
    "average_revenue_per_transaction": float(final_sales_dataset["revenue"].mean()),
    "total_returns": int(final_sales_dataset["returns"].sum()),
    "return_rate": float((final_sales_dataset["returns"].sum() / final_sales_dataset["units_sold"].replace(0, np.nan).sum()) * 100),
    "total_customers": int(final_sales_dataset["customer_id"].nunique()),
    "total_products": int(final_sales_dataset["product_id"].nunique()),
    "total_sites": int(final_sales_dataset["site_id"].nunique()),
    "total_stockout_items": int((inventory["stockout_flag"].str.lower() == "yes").sum()),
}])

# 8.3 sales_monthly_trend
sales_monthly_trend = final_sales_dataset.groupby(["year", "month", "month_name"], as_index=False).agg(
    total_sales_records=("product_id", "count"),
    total_units_sold=("units_sold", "sum"),
    total_revenue=("revenue", "sum"),
    total_discounts=("discounts", "sum"),
    total_net_revenue=("net_revenue", "sum"),
    estimated_profit=("estimated_profit", "sum"),
    total_returns=("returns", "sum"),
)
sales_monthly_trend["average_revenue_per_transaction"] = safe_divide(sales_monthly_trend["total_revenue"], sales_monthly_trend["total_sales_records"])
sales_monthly_trend["year_month"] = sales_monthly_trend["year"].astype(str) + "-" + sales_monthly_trend["month"].astype(str).str.zfill(2)
sales_monthly_trend = sales_monthly_trend[[
    "year", "month", "month_name", "year_month", "total_sales_records", "total_units_sold",
    "total_revenue", "total_discounts", "total_net_revenue", "estimated_profit", "total_returns",
    "average_revenue_per_transaction"
]].sort_values(["year", "month"])

# 8.4 product_performance
product_sales = final_sales_dataset.groupby([
    "product_id", "product_name", "category", "subcategory", "supplier", "unit_cost", "unit_price", "shelf_life"
], as_index=False).agg(
    total_sales_records=("product_id", "count"),
    total_units_sold=("units_sold", "sum"),
    total_revenue=("revenue", "sum"),
    total_discounts=("discounts", "sum"),
    total_net_revenue=("net_revenue", "sum"),
    estimated_profit=("estimated_profit", "sum"),
    total_returns=("returns", "sum"),
)
inv_by_product = inventory.groupby("product_id", as_index=False).agg(
    current_ending_inventory=("ending_inventory", "sum"),
    total_replenishment=("replenishment", "sum"),
    stockout_count=("stockout_flag", lambda x: int((x.str.lower() == "yes").sum()))
)
product_performance = product_sales.merge(inv_by_product, on="product_id", how="left")
product_performance["return_rate"] = safe_divide(product_performance["total_returns"], product_performance["total_units_sold"]) * 100
product_performance["sales_to_stock_ratio"] = safe_divide(product_performance["total_units_sold"], product_performance["current_ending_inventory"])
product_performance["stock_status"] = product_performance.apply(lambda r: stock_status_rule({
    "total_ending_inventory": r["current_ending_inventory"],
    "total_units_sold": r["total_units_sold"],
    "stockout_count": r["stockout_count"],
    "sales_to_stock_ratio": r["sales_to_stock_ratio"],
}), axis=1)
product_performance["product_rank"] = product_performance["total_net_revenue"].rank(method="dense", ascending=False).astype(int)
product_performance = product_performance.sort_values("product_rank")

# 8.5 inventory_monitoring
inventory_monitoring = inventory.merge(products, on="product_id", how="left")
sales_by_product = final_sales_dataset.groupby("product_id", as_index=False).agg(total_units_sold=("units_sold", "sum"))
inventory_monitoring = inventory_monitoring.groupby([
    "product_id", "product_name", "category", "subcategory", "supplier"
], as_index=False).agg(
    total_beginning_inventory=("beginning_inventory", "sum"),
    total_ending_inventory=("ending_inventory", "sum"),
    total_replenishment=("replenishment", "sum"),
    stockout_count=("stockout_flag", lambda x: int((x.str.lower() == "yes").sum()))
)
inventory_monitoring = inventory_monitoring.merge(sales_by_product, on="product_id", how="left")
inventory_monitoring["total_units_sold"] = inventory_monitoring["total_units_sold"].fillna(0)
inventory_monitoring["inventory_change"] = inventory_monitoring["total_ending_inventory"] - inventory_monitoring["total_beginning_inventory"]
inventory_monitoring["stockout_rate"] = safe_divide(inventory_monitoring["stockout_count"], pd.Series([len(inventory)] * len(inventory_monitoring))) * 100
inventory_monitoring["sales_to_stock_ratio"] = safe_divide(inventory_monitoring["total_units_sold"], inventory_monitoring["total_ending_inventory"])
inventory_monitoring["stock_status"] = inventory_monitoring.apply(stock_status_rule, axis=1)
inventory_monitoring["restock_priority"] = inventory_monitoring.apply(restock_priority_rule, axis=1)

# 8.6 customer_analysis
customer_analysis = final_sales_dataset.groupby([
    "customer_id", "age", "gender", "income_bracket", "purchase_frequency", "average_spend"
], as_index=False).agg(
    total_sales_records=("product_id", "count"),
    total_units_bought=("units_sold", "sum"),
    total_revenue=("revenue", "sum"),
    total_net_revenue=("net_revenue", "sum"),
    total_returns=("returns", "sum"),
)
customer_analysis["customer_value_segment"] = value_segment(customer_analysis["total_net_revenue"])
customer_analysis["buying_frequency_segment"] = np.select(
    [customer_analysis["purchase_frequency"] <= 3,
     (customer_analysis["purchase_frequency"] > 3) & (customer_analysis["purchase_frequency"] <= 7),
     customer_analysis["purchase_frequency"] > 7],
    ["low_frequency", "medium_frequency", "high_frequency"],
    default="medium_frequency"
)

# 8.7 promotion_performance
promo_sales = final_sales_dataset[final_sales_dataset["active_promotion_id"].notna()].copy()
if len(promo_sales) > 0:
    promotion_performance = promo_sales.groupby([
        "active_promotion_id", "product_id", "product_name", "category", "discount_type", "promotion_discount_amount"
    ], as_index=False).agg(
        total_sales_records=("product_id", "count"),
        total_units_sold=("units_sold", "sum"),
        total_revenue=("revenue", "sum"),
        total_discounts=("discounts", "sum"),
        total_net_revenue=("net_revenue", "sum"),
    ).rename(columns={"active_promotion_id": "promotion_id", "promotion_discount_amount": "discount_amount"})
    promo_dates = promos[["promotion_id", "start_date", "end_date"]]
    promotion_performance = promotion_performance.merge(promo_dates, on="promotion_id", how="left")
else:
    promotion_performance = promos.merge(products[["product_id", "product_name", "category"]], on="product_id", how="left")
    promotion_performance = promotion_performance.rename(columns={"discount_amount": "discount_amount"})
    promotion_performance["total_sales_records"] = 0
    promotion_performance["total_units_sold"] = 0
    promotion_performance["total_revenue"] = 0.0
    promotion_performance["total_discounts"] = 0.0
    promotion_performance["total_net_revenue"] = 0.0

promotion_performance["promotion_status"] = np.where(
    pd.to_datetime(promotion_performance["end_date"]) < pd.Timestamp.today().normalize(),
    "ended",
    "active_or_future"
)
promotion_performance["effectiveness_status"] = np.select(
    [promotion_performance["total_net_revenue"] > promotion_performance["total_net_revenue"].quantile(0.66),
     promotion_performance["total_net_revenue"] < promotion_performance["total_net_revenue"].quantile(0.33)],
    ["high_effective", "low_effective"],
    default="medium_effective"
)
promotion_performance = promotion_performance[[
    "promotion_id", "product_id", "product_name", "category", "start_date", "end_date", "discount_type", "discount_amount",
    "total_sales_records", "total_units_sold", "total_revenue", "total_discounts", "total_net_revenue",
    "promotion_status", "effectiveness_status"
]]

# 8.8 logistics_performance
logistics_performance = logistics.groupby(["delivery_status", "transportation_type"], as_index=False).agg(
    total_shipments=("shipment_id", "count"),
    total_quantity_shipped=("quantity", "sum"),
)
logistics_performance["delivered_shipments"] = np.where(logistics_performance["delivery_status"].str.lower() == "delivered", logistics_performance["total_shipments"], 0)
logistics_performance["delayed_shipments"] = np.where(logistics_performance["delivery_status"].str.lower() == "delayed", logistics_performance["total_shipments"], 0)
logistics_performance["cancelled_shipments"] = np.where(logistics_performance["delivery_status"].str.lower() == "cancelled", logistics_performance["total_shipments"], 0)
total_shipments_global = logistics_performance["total_shipments"].sum()
logistics_performance["delivered_rate"] = safe_divide(logistics_performance["delivered_shipments"], pd.Series([total_shipments_global] * len(logistics_performance))) * 100
logistics_performance["delayed_rate"] = safe_divide(logistics_performance["delayed_shipments"], pd.Series([total_shipments_global] * len(logistics_performance))) * 100
logistics_performance["cancelled_rate"] = safe_divide(logistics_performance["cancelled_shipments"], pd.Series([total_shipments_global] * len(logistics_performance))) * 100

# 8.9 seasonal_planning_performance
planning_temp = planning.copy()
planning_temp["month_dt"] = pd.to_datetime(planning_temp["month_date"])
planning_temp["year"] = planning_temp["month_dt"].dt.year
planning_temp["month_num"] = planning_temp["month_dt"].dt.month
planning_temp["month_name"] = planning_temp["month_dt"].dt.month_name()
seasonal_planning_performance = planning_temp.groupby([
    "month_date", "year", "month_num", "month_name", "product_category"
], as_index=False).agg(
    total_forecasted_sales=("forecasted_sales", "sum"),
    total_actual_sales=("actual_sales", "sum"),
    avg_seasonal_adjustment=("seasonal_adjustments", "mean"),
)
seasonal_planning_performance = seasonal_planning_performance.rename(columns={"month_num": "month"})
seasonal_planning_performance["forecast_error"] = seasonal_planning_performance["total_actual_sales"] - seasonal_planning_performance["total_forecasted_sales"]
seasonal_planning_performance["forecast_accuracy"] = 100 - (abs(seasonal_planning_performance["forecast_error"]) / seasonal_planning_performance["total_forecasted_sales"].replace(0, np.nan) * 100)
seasonal_planning_performance["forecast_accuracy"] = seasonal_planning_performance["forecast_accuracy"].replace([np.inf, -np.inf], np.nan).fillna(0)
seasonal_planning_performance["planning_status"] = np.where(
    abs(seasonal_planning_performance["forecast_error"]) <= seasonal_planning_performance["total_forecasted_sales"] * 0.1,
    "accurate",
    "needs_review"
)

# 8.10 etl_run_log
etl_run_log = pd.DataFrame([{
    "run_name": RUN_NAME,
    "source_dataset": "Retail Insights Sales Inventory Customer etc",
    "start_time": datetime.now().isoformat(),
    "end_time": datetime.now().isoformat(),
    "status": "success",
    "rows_extracted": int(sum(len(df) for df in raw_tables.values())),
    "rows_transformed": int(sum(len(df) for df in [
        final_sales_dataset, owner_kpi_summary, sales_monthly_trend, product_performance,
        inventory_monitoring, customer_analysis, promotion_performance, logistics_performance,
        seasonal_planning_performance
    ])),
    "rows_loaded": 0,
    "error_count": 0,
    "notes": "ETL completed and final app tables exported as CSV."
}])

Pembentukkan data mart, yaitu tabel siap pakai untuk dashboard BI. Kode membuat `final_sales_dataset` sebagai data sales terintegrasi, lalu membuat ringkasan seperti KPI owner, tren penjualan bulanan, performa produk, monitoring inventory, analisis customer, performa promosi, performa logistics, dan evaluasi seasonal planning. Tujuannya agar dashboard tidak perlu menghitung ulang data dari tabel mentah.

### **Validation check**

In [ ]:

validation_checks = []
validation_checks.append({"check_name": "sales_raw_rows", "value": len(sales_raw), "status": "ok" if len(sales_raw) > 0 else "failed"})
validation_checks.append({"check_name": "final_sales_rows", "value": len(final_sales_dataset), "status": "ok" if len(final_sales_dataset) == len(sales) else "warning"})
validation_checks.append({"check_name": "missing_product_after_join", "value": int(final_sales_dataset["product_name"].isna().sum()), "status": "ok" if final_sales_dataset["product_name"].isna().sum() == 0 else "warning"})
validation_checks.append({"check_name": "missing_customer_after_join", "value": int(final_sales_dataset["gender"].isna().sum()), "status": "ok" if final_sales_dataset["gender"].isna().sum() == 0 else "warning"})
validation_checks.append({"check_name": "missing_site_after_join", "value": int(final_sales_dataset["site_name"].isna().sum()), "status": "ok" if final_sales_dataset["site_name"].isna().sum() == 0 else "warning"})
validation_checks.append({"check_name": "negative_net_revenue", "value": int((final_sales_dataset["net_revenue"] < 0).sum()), "status": "ok" if (final_sales_dataset["net_revenue"] < 0).sum() == 0 else "warning"})
validation_report = pd.DataFrame(validation_checks)

print("\nValidation report:")
display(validation_report) if "display" in globals() else print(validation_report)


Validation report:
                    check_name  value status
0               sales_raw_rows    219     ok
1             final_sales_rows    219     ok
2   missing_product_after_join      0     ok
3  missing_customer_after_join      0     ok
4      missing_site_after_join      0     ok
5         negative_net_revenue      0     ok


validation check adalah proses untuk memastikan hasil ETL tidak bermasalah. Pengecekan dilakukan pada jumlah data sales, hasil join dengan product, customer, dan site, serta nilai `net_revenue` negatif. Tujuannya untuk memastikan data hasil transformasi tetap lengkap, valid, dan layak digunakan untuk dashboard.

# **LOAD**

In [ ]:

# LOAD / EXPORT ETL OUTPUTS TO CSV
from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/TUGAS SMESTER 4/BI/dataset/CLEANED")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. EXPORT CLEANED DATASET TABLES

CLEANED_DIR = OUTPUT_DIR / "cleaned_tables"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

cleaned_tables = {
    "clean_sales": sales,
    "clean_products": products,
    "clean_inventory": inventory,
    "clean_customers": customers,
    "clean_sites": sites,
    "clean_logistics": logistics,
    "clean_promotions": promos,
    "clean_planning": planning,
}

for table_name, df in cleaned_tables.items():
    output_path = CLEANED_DIR / f"{table_name}.csv"
    df.to_csv(output_path, index=False)
    print(f"Exported {table_name}: {df.shape} -> {output_path}")


# 2. EXPORT DATA MART / FINAL APP TABLES

data_mart_tables = {
    "final_sales_dataset": final_sales_dataset,
    "owner_kpi_summary": owner_kpi_summary,
    "sales_monthly_trend": sales_monthly_trend,
    "product_performance": product_performance,
    "inventory_monitoring": inventory_monitoring,
    "customer_analysis": customer_analysis,
    "promotion_performance": promotion_performance,
    "logistics_performance": logistics_performance,
    "seasonal_planning_performance": seasonal_planning_performance,
    "etl_run_log": etl_run_log,
    "validation_report": validation_report,
}

for table_name, df in data_mart_tables.items():
    output_path = OUTPUT_DIR / f"{table_name}.csv"
    df.to_csv(output_path, index=False)
    print(f"Exported {table_name}: {df.shape} -> {output_path}")

# 3. EXPORT DATA WAREHOUSE TABLES

WAREHOUSE_DIR = OUTPUT_DIR / "warehouse_tables"
WAREHOUSE_DIR.mkdir(parents=True, exist_ok=True)

warehouse_tables = {
    "dw_dim_date": dim_date,
    "dw_dim_product": dim_product,
    "dw_dim_customer": dim_customer,
    "dw_dim_site": dim_site,
    "dw_dim_promotion": dim_promotion,
    "dw_fact_sales": fact_sales,
    "dw_fact_inventory": fact_inventory,
    "dw_fact_logistics": fact_logistics,
    "dw_fact_promotion": fact_promotion,
    "dw_fact_seasonal_planning": fact_seasonal_planning,
}

for table_name, df in warehouse_tables.items():
    output_path = WAREHOUSE_DIR / f"{table_name}.csv"
    df.to_csv(output_path, index=False)
    print(f"Exported {table_name}: {df.shape} -> {output_path}")


print("\nETL finished successfully.")
print(f"Cleaned dataset CSV files are saved in: {CLEANED_DIR}")
print(f"Data mart / final app CSV files are saved in: {OUTPUT_DIR}")
print(f"Data warehouse CSV files are saved in: {WAREHOUSE_DIR}")

Exported clean_sales: (219, 15) -> /content/drive/MyDrive/TUGAS SMESTER 4/BI/dataset/CLEANED/cleaned_tables/clean_sales.csv
Exported clean_products: (100, 8) -> /content/drive/MyDrive/TUGAS SMESTER 4/BI/dataset/CLEANED/cleaned_tables/clean_products.csv
Exported clean_inventory: (100, 6) -> /content/drive/MyDrive/TUGAS SMESTER 4/BI/dataset/CLEANED/cleaned_tables/clean_inventory.csv
Exported clean_customers: (50, 6) -> /content/drive/MyDrive/TUGAS SMESTER 4/BI/dataset/CLEANED/cleaned_tables/clean_customers.csv
Exported clean_sites: (50, 9) -> /content/drive/MyDrive/TUGAS SMESTER 4/BI/dataset/CLEANED/cleaned_tables/clean_sites.csv
Exported clean_logistics: (150, 7) -> /content/drive/MyDrive/TUGAS SMESTER 4/BI/dataset/CLEANED/cleaned_tables/clean_logistics.csv
Exported clean_promotions: (50, 7) -> /content/drive/MyDrive/TUGAS SMESTER 4/BI/dataset/CLEANED/cleaned_tables/clean_promotions.csv
Exported clean_planning: (120, 7) -> /content/drive/MyDrive/TUGAS SMESTER 4/BI/dataset/CLEANED/cleane

Setelah data dibersihkan dan melalui proses data warehouse, langkah terakhir adalah menyimpan hasil akhir ETL ke file CSV. Output dipisahkan menjadi cleaned tables, data mart tables, dan warehouse tables. Tujuannya agar data hasil cleaning, struktur data warehouse, dan tabel dashboard tersimpan rapi serta bisa digunakan kembali oleh aplikasi BI.